# 🅿️ 2GIS Parking Parser — Almaty

**Стратегия:**
Используем официальный **2GIS Places API** (`catalog.api.2gis.com/3.0/items`), а не парсинг HTML.

**Почему API, а не Selenium/Playwright?**
- 2GIS — SPA на React, данные подгружаются асинхронно через внутренний API
- API возвращает чистый JSON с десятками полей сразу
- В 10+ раз быстрее, нет проблем с headless-детектом
- Официально задокументировано, стабильно

**Ограничение demo-ключа:** max 5 страниц × 10 объектов = 50 результатов на один запрос (точку).  
**Решение:** равномерная сетка из 82 точек по Алматы с шагом ~3×2.5 км (проверено: даёт 611 уникальных парковок против ~410 при грубой сетке по районам). Дедупликация по `id`.

**Нужно:**
1. Зарегистрируйся на [platform.2gis.ru](https://platform.2gis.ru) → создай Demo Key (бесплатно, 1000 запросов)
2. Создай Google Sheets и настрой Google Sheets API (инструкция в ячейке ниже)
3. Вставь ключи в ячейку конфигурации

## 📦 Установка зависимостей

In [1]:
!pip install requests pandas gspread google-auth tqdm

## ⚙️ Конфигурация

### Получение 2GIS API ключа:
1. Идёшь на [platform.2gis.ru](https://platform.2gis.ru)
2. Регистрируешься (email или через 2GIS аккаунт)
3. Dashboard → "Создать демо-ключ" → называешь как угодно
4. Копируешь ключ сюда

### Получение Google Sheets credentials:
1. Идёшь на [console.cloud.google.com](https://console.cloud.google.com)
2. Создаёшь проект → включаешь Google Sheets API и Google Drive API
3. IAM → Service Accounts → Create → скачиваешь JSON
4. Кладёшь JSON рядом с ноутбуком, называешь `service_account.json`
5. Создаёшь Google Sheets и шаришь его на email из JSON (`client_email`)

In [2]:
# ============================================================
# КОНФИГУРАЦИЯ — заполни перед запуском
# ============================================================

TWOGIS_API_KEY = "69319bc5-e7fa-4fd6-bf55-9fee36a7b7c9"          # ключ с platform.2gis.ru
GOOGLE_CREDENTIALS_FILE = "C:\\Users\\User\\Desktop\\Sergek TE\\2GIS_Parking_Parser\\ste-2gis-parking-parser-03c14e1e906a.json"  # путь к JSON сервисного аккаунта
GOOGLE_SHEET_NAME = "2GIS Parking Almaty"    # название таблицы в Google Sheets

# ============================================================

## 📍 Сетка точек по Алматы

Алматы — большой город (~150 км²). Одним запросом в центр не поймать парковки на окраинах.
Разбиваем на сетку: центр + 8 районов + несколько дополнительных зон (БЦ, ТРЦ, аэропорт).

In [3]:
import requests
import pandas as pd
import numpy as np
import json
import time
import re
from tqdm import tqdm
from datetime import datetime

# Сетка точек для покрытия всего Алматы.
# ПРОВЕРЕНО: равномерная сетка с шагом ~0.04°/0.025° (радиус 1500м на запрос)
# даёт 82 точки и покрывает существенно больше объектов, чем 18 ручных точек
# по районам (611 уникальных парковок против изначальных ~410 при той же
# логике дедупликации). Граница Алматы: lon 76.78–77.08, lat 43.17–43.40.
lons = np.arange(76.78, 77.08, 0.04)
lats = np.arange(43.17, 43.40, 0.025)

ALMATY_GRID = [
    {"name": f"grid_{round(lo,3)}_{round(la,3)}", "lon": round(lo, 4), "lat": round(la, 4)}
    for lo in lons
    for la in lats
]

# Точки, которые сетка может зацепить неточно (вынесенные объекты)
ALMATY_GRID += [
    {"name": "Аэропорт", "lon": 77.0207, "lat": 43.3521},
    {"name": "MEGA",     "lon": 76.8280, "lat": 43.2215},
]

print(f"Сетка точек: {len(ALMATY_GRID)} локаций")

Сетка точек: 82 локаций


## 🔍 Функции для работы с 2GIS API

In [4]:
BASE_URL = "https://catalog.api.2gis.com/3.0/items"

# Все доступные поля для обогащения данных
ALL_FIELDS = ",".join([
    "items.point",
    "items.address",
    "items.schedule",
    "items.attribute_groups",
    "items.org",
    "items.rubrics",
    "items.links",
    "items.contact_groups",
    "items.name_ex",
    "items.geometry",
    "items.reviews",
    "items.external_content",
    "items.description",
    "items.flags",
])


def fetch_parkings_near_point(lon: float, lat: float, radius: int = 3000,
                               page_size: int = 10) -> list[dict]:
    """Получаем все парковки в радиусе radius метров от точки (lon, lat).
    Пагинируем до 5 страниц (лимит demo-ключа: max page_size=10, max 5 страниц)."""
    results = []
    
    for page in range(1, 6):  # демо-ключ: max 5 страниц
        params = {
            "q": "парковка",
            "rubric_id": "60340",     # рубрика «Парковки» в 2GIS
            "point": f"{lon},{lat}",
            "radius": radius,
            "sort": "distance",
            "page_size": page_size,
            "page": page,
            "fields": ALL_FIELDS,
            "key": TWOGIS_API_KEY,
        }
        
        try:
            resp = requests.get(BASE_URL, params=params, timeout=15)
            resp.raise_for_status()
            data = resp.json()
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  Ошибка запроса (стр.{page}): {e}")
            break
        except json.JSONDecodeError:
            print(f"  ⚠️  Невалидный JSON на стр.{page}")
            break
        
        meta = data.get("meta", {})
        if meta.get("code") != 200:
            print(f"  ⚠️  API вернул код {meta.get('code')}: {meta.get('error', {}).get('message', '')}")
            break
        
        items = data.get("result", {}).get("items", [])
        if not items:
            break  # нет больше результатов
        
        results.extend(items)
        
        total = data.get("result", {}).get("total", 0)
        if len(results) >= total or page * page_size >= total:
            break
        
        time.sleep(0.3)  # уважаем rate limit
    
    return results


print("✅ Функция fetch_parkings_near_point готова")

✅ Функция fetch_parkings_near_point готова


## 🔬 ДИАГНОСТИКА: смотрим реальную структуру ответа API

**Важно!** Прежде чем парсить 600+ объектов, проверяем на ОДНОМ реальном объекте,
какие поля API реально возвращает. Предыдущая версия парсера угадывала структуру
(`address.components`, `attribute_groups`) — и угадала неверно, из-за чего адрес,
тариф и расписание остались пустыми.

Запусти эту ячейку и посмотри на вывод — это покажет, какие поля реально есть.

In [ ]:
# Делаем один тестовый запрос и смотрим ПОЛНЫЙ сырой JSON
test_params = {
    "q": "парковка",
    "rubric_id": "60340",
    "point": "76.9285,43.2567",
    "radius": 3000,
    "page_size": 3,
    "page": 1,
    "fields": ALL_FIELDS,
    "key": TWOGIS_API_KEY,
}

test_resp = requests.get(BASE_URL, params=test_params, timeout=15)
test_data = test_resp.json()

test_items = test_data.get("result", {}).get("items", [])
print(f"Получено {len(test_items)} тестовых объектов\n")

if test_items:
    print("=== ПОЛНЫЙ JSON ПЕРВОГО ОБЪЕКТА ===\n")
    print(json.dumps(test_items[0], ensure_ascii=False, indent=2))
else:
    print("⚠️ Нет результатов, проверь параметры запроса")

Получено 3 тестовых объектов

=== ПОЛНЫЙ JSON ПЕРВОГО ОБЪЕКТА ===

{
  "address": {
    "building_code": "A20C0Y1",
    "building_id": "9430047375069375",
    "building_name": "ТЦ Алтын-Тараз",
    "components": [
      {
        "number": "3",
        "street": "проспект Абылай хана",
        "street_id": "9430150454182643",
        "type": "street_number"
      }
    ],
    "postcode": "050050"
  },
  "address_name": "проспект Абылай хана, 3",
  "attribute_groups": [
    {
      "attributes": [
        {
          "id": "70000201006755982",
          "name": "Бесплатно 1 час",
          "tag": "parking_free_parking"
        },
        {
          "id": "70000201006756660",
          "name": "Крытая",
          "tag": "parking_indoor_parking"
        },
        {
          "id": "70000201006755983",
          "name": "Тёплая",
          "tag": "parking_warm_parking"
        },
        {
          "id": "70000201006755980",
          "name": "​ Первый час-бесплатно, от 1 до 12 часов-10

In [6]:
# Дополнительно: смотрим ключи верхнего уровня по ВСЕМ тестовым объектам,
# плюс отдельно структуру schedule и attribute_groups если они есть хоть у одного
print("=== Ключи верхнего уровня (по всем тестовым объектам) ===")
all_keys = set()
for item in test_items:
    all_keys.update(item.keys())
print(sorted(all_keys))

print("\n=== Объекты со schedule ===")
for item in test_items:
    if item.get("schedule"):
        print(f"\n--- {item.get('name')} ---")
        print(json.dumps(item["schedule"], ensure_ascii=False, indent=2))

print("\n=== Объекты со attribute_groups ===")
for item in test_items:
    if item.get("attribute_groups"):
        print(f"\n--- {item.get('name')} ---")
        print(json.dumps(item["attribute_groups"], ensure_ascii=False, indent=2))

print("\n=== Адресные поля (address_name / address_comment / address) ===")
for item in test_items:
    print(f"{item.get('name')!r}: address_name={item.get('address_name')!r}, "
          f"address_comment={item.get('address_comment')!r}, "
          f"address={item.get('address')!r}")

=== Ключи верхнего уровня (по всем тестовым объектам) ===
['address', 'address_name', 'attribute_groups', 'flags', 'full_name', 'id', 'links', 'name', 'name_ex', 'org', 'point', 'purpose_name', 'reviews', 'rubrics', 'schedule', 'type']

=== Объекты со schedule ===

--- Автостоянка ---
{
  "Fri": {
    "working_hours": [
      {
        "from": "00:00",
        "to": "24:00"
      }
    ]
  },
  "Mon": {
    "working_hours": [
      {
        "from": "00:00",
        "to": "24:00"
      }
    ]
  },
  "Sat": {
    "working_hours": [
      {
        "from": "00:00",
        "to": "24:00"
      }
    ]
  },
  "Sun": {
    "working_hours": [
      {
        "from": "00:00",
        "to": "24:00"
      }
    ]
  },
  "Thu": {
    "working_hours": [
      {
        "from": "00:00",
        "to": "24:00"
      }
    ]
  },
  "Tue": {
    "working_hours": [
      {
        "from": "00:00",
        "to": "24:00"
      }
    ]
  },
  "Wed": {
    "working_hours": [
      {
        "from": "00:00

## 🗺️ Парсинг по всей сетке Алматы

In [7]:
all_raw = {}  # id -> item (автоматическая дедупликация)

for point in tqdm(ALMATY_GRID, desc="Обход точек"):
    items = fetch_parkings_near_point(point["lon"], point["lat"])
    new_count = 0
    for item in items:
        item_id = item.get("id")
        if item_id and item_id not in all_raw:
            all_raw[item_id] = item
            new_count += 1
    print(f"  📍 {point['name']}: +{new_count} новых (всего уникальных: {len(all_raw)})")
    time.sleep(0.5)

print(f"\n🎉 Итого уникальных парковок: {len(all_raw)}")

Обход точек:   0%|          | 0/82 [00:00<?, ?it/s]

  ⚠️  API вернул код 404: Results not found
  📍 grid_76.78_43.17: +0 новых (всего уникальных: 0)


Обход точек:   1%|          | 1/82 [00:00<01:05,  1.24it/s]

  ⚠️  API вернул код 404: Results not found
  📍 grid_76.78_43.195: +0 новых (всего уникальных: 0)


Обход точек:   2%|▏         | 2/82 [00:01<01:04,  1.25it/s]

...

  📍 Аэропорт: +0 новых (всего уникальных: 609)


Обход точек:  99%|█████████▉| 81/82 [01:59<00:00,  1.21it/s]

  📍 MEGA: +0 новых (всего уникальных: 609)


Обход точек: 100%|██████████| 82/82 [02:03<00:00,  1.51s/it]


🎉 Итого уникальных парковок: 609


## 🔧 Парсинг и нормализация данных

In [ ]:
def parse_schedule(schedule: dict | None) -> dict:
    """Парсим расписание работы.
    
    РЕАЛЬНАЯ структура (проверено диагностикой на живых данных):
    schedule = {"Mon": {"working_hours": [{"from": "00:00", "to": "24:00"}]}, "Fri": {...}, ...}
    Это словарь по дням недели (3-буквенные ключи), НЕ список с полем 'items' как
    предполагалось раньше. И 24-часовая работа кодируется как from=00:00 to=24:00,
    а не to=00:00 (старая версия эту запись не ловила вообще).
    """
    if not schedule:
        return {"hours_raw": "", "is_24h": False, "works_on_weekends": None}
    
    days_map = {
        "Mon": "Пн", "Tue": "Вт", "Wed": "Ср",
        "Thu": "Чт", "Fri": "Пт", "Sat": "Сб", "Sun": "Вс"
    }
    day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
    
    working_hours_str = []
    days_24h = []
    days_present = set()
    
    for day_key in day_order:
        day_data = schedule.get(day_key)
        if not day_data:
            continue
        
        days_present.add(day_key)
        day_ru = days_map[day_key]
        working = day_data.get("working_hours", [])
        
        for wh in working:
            frm = wh.get("from", "")
            to = wh.get("to", "")
            # Круглосуточно кодируется как 00:00–24:00 (НЕ 00:00–00:00)
            if frm == "00:00" and to == "24:00":
                days_24h.append(day_ru)
                working_hours_str.append(f"{day_ru}: 24ч")
            else:
                working_hours_str.append(f"{day_ru}: {frm}–{to}")
    
    is_24h = len(days_24h) == len(days_present) and len(days_present) >= 7
    works_on_weekends = bool({"Sat", "Sun"} & days_present)
    
    return {
        "hours_raw": " | ".join(working_hours_str),
        "is_24h": is_24h,
        "works_on_weekends": works_on_weekends,
    }


def parse_tariff(attribute_groups: list) -> dict:
    """
    Разбираем атрибуты парковки на тарифные компоненты.
    
    РЕАЛЬНАЯ структура (проверено диагностикой):
    attribute_groups = [
        {"name": "Паркинги", "attributes": [
            {"name": "Бесплатно 1 час", "tag": "parking_free_parking"},
            {"name": "Крытая", "tag": "parking_indoor_parking"},
            {"name": "​ Первый час-бесплатно, от 1 до 12 часов-100 тг...",
             "tag": "parking_price_comment"},
            ...
        ]},
        {"name": "Способы оплаты", "attributes": [...]},
    ]
    Атрибуты — это булевы метки-теги (наличие = True), а не name/value пары.
    Цена не структурирована — она внутри свободного текста tag='parking_price_comment',
    поэтому парсим её regex'ом.
    """
    result = {
        "is_paid": None,
        "tariff_raw": "",
        "price_hourly": None,
        "price_daily": None,
        "price_monthly": None,
        "first_hour_free": False,
        "parking_type": "",        # наземная / подземная / многоуровневая
        "covered": None,           # крытая
        "heated": None,            # тёплая
        "security": None,          # охраняемая
        "video_surveillance": None,
        "disabled_spaces": None,
        "ev_charging": None,
        "payment_methods": [],     # наличные / карта / QR / приложение
    }
    
    if not attribute_groups:
        return result
    
    # Карта известных тегов -> что они означают
    tag_map = {
        "parking_free_parking": ("is_paid", False),
        "parking_paid_parking": ("is_paid", True),
        "parking_indoor_parking": ("covered", True),
        "parking_outdoor_parking": ("covered", False),
        "parking_warm_parking": ("heated", True),
        "parking_underground_parking": ("parking_type", "Подземная"),
        "parking_multilevel_parking": ("parking_type", "Многоуровневая"),
        "parking_guarded": ("security", True),
        "parking_security": ("security", True),
        "parking_video_surveillance": ("video_surveillance", True),
        "parking_disabled_access": ("disabled_spaces", True),
        "parking_ev_charging": ("ev_charging", True),
    }
    
    payment_tags = {
        "general_payment_type_cash": "Наличные",
        "general_payment_type_cards": "Карта",
        "general_payment_type_qrcode": "QR-код",
    }
    
    price_comment = ""
    
    for group in attribute_groups:
        for attr in group.get("attributes", []):
            tag = attr.get("tag", "")
            attr_name = attr.get("name", "")
            
            if tag == "parking_free_parking" and "1 час" in attr_name.lower():
                # "Бесплатно 1 час" -> первый час бесплатно (не вся парковка бесплатная)
                result["first_hour_free"] = True
                continue
            
            if tag in tag_map:
                field, val = tag_map[tag]
                result[field] = val
                continue
            
            if tag == "parking_price_comment":
                price_comment = attr_name.strip()
                continue
            
            if tag in payment_tags:
                result["payment_methods"].append(payment_tags[tag])
    
    # Если явно не указана платность, но есть платный комментарий по цене — считаем платной
    if result["is_paid"] is None and price_comment:
        result["is_paid"] = True
    
    result["tariff_raw"] = price_comment
    result["payment_methods"] = ", ".join(result["payment_methods"])
    
    # Парсим цены из свободного текста типа:
    # "Первый час-бесплатно, от 1 до 12 часов-100 тг, от 12 до 24 часов-500 тг."
    if price_comment:
        text = price_comment.lower()
        
        if "первый час" in text and "бесплат" in text.split(",")[0]:
            result["first_hour_free"] = True
        
        # Цена за час: ищем паттерн "час...NNN тг" / "час - NNN тг"
        hourly_matches = re.findall(r'час[а-я]*[\s\-–—]*(\d+)\s*тг', text)
        if hourly_matches:
            result["price_hourly"] = int(hourly_matches[0])
        
        # Цена за сутки/день
        daily_matches = re.findall(r'(?:сутки|сутк|день)[а-я]*[\s\-–—]*(\d+)\s*тг', text)
        if daily_matches:
            result["price_daily"] = int(daily_matches[0])
        
        # Абонемент/месяц
        monthly_matches = re.findall(r'месяц[а-я]*[\s\-–—]*(\d+)\s*тг', text)
        if monthly_matches:
            result["price_monthly"] = int(monthly_matches[0])
    
    return result


def detect_parking_category(name: str, rubrics: list) -> str:
    """Определяем тип парковки: городская / ТЦ / БЦ / ЖК / отель / аэропорт / частная."""
    name_lower = name.lower()
    rubric_names = " ".join([r.get("name", "").lower() for r in rubrics])
    combined = name_lower + " " + rubric_names
    
    if any(w in combined for w in ["аэропорт", "airport"]):
        return "Аэропорт"
    if any(w in combined for w in ["мега", "алтын", "достык", "есентай", "апорт",
                                    "рамстор", "молл", "mall", "plaza", "плаза",
                                    "торгов", "тц", "трц", "трк"]):
        return "ТЦ/ТРЦ"
    if any(w in combined for w in ["бизнес", "офис", "бц", "центр", "tower", "тауэр",
                                    "бурж", "business"]):
        return "БЦ"
    if any(w in combined for w in ["жилой", "жк", "residential", "комплекс", "квартал"]):
        return "ЖК"
    if any(w in combined for w in ["отель", "гостиниц", "hotel", "хаятт", "мариотт",
                                    "хилтон", "sheraton"]):
        return "Отель"
    if any(w in combined for w in ["больниц", "клиник", "госпиталь", "медицин"]):
        return "Медучреждение"
    if any(w in combined for w in ["муниципальн", "городск", "коммунальн"]):
        return "Городская"
    return "Частная/иная"


def build_2gis_url(item: dict) -> str:
    """Строим прямую ссылку на объект в 2GIS."""
    item_id = item.get("id", "")
    point = item.get("point", {})
    lon = point.get("lon", "")
    lat = point.get("lat", "")
    if item_id and lon and lat:
        return f"https://2gis.kz/almaty/firm/{item_id}/{lon},{lat}"
    return ""


print("✅ Функции парсинга готовы")

✅ Функции парсинга готовы


## 🏗️ Сборка DataFrame

In [71]:
rows = []

for item_id, item in tqdm(all_raw.items(), desc="Нормализация"):
    try:
        # --- Базовые поля ---
        name = item.get("name", "")
        full_name = item.get("name_ex", {}).get("primary", name)
        
        # Адрес — РЕАЛЬНАЯ структура: address_name уже готовая строка
        # "проспект Абылай хана, 3". address.components[].type == 'street_number'
        # содержит И название улицы (поле street), И номер дома (поле number) вместе —
        # тип 'street' отдельно не существует, поэтому старая версия ничего не находила.
        address_obj = item.get("address", {}) or {}
        address_full = item.get("address_name") or ""
        components = address_obj.get("components", [])
        street = ""
        building = ""
        for comp in components:
            if comp.get("type") == "street_number":
                street = comp.get("street", "")
                building = comp.get("number", "")
        # Район/город — иногда приходит как components[].type == 'location' с полем comment
        district = ""
        for comp in components:
            if comp.get("type") == "location":
                district = comp.get("comment", "")
        # Фоллбэк: если нет ни address_name, ни street_number — берём landmark/building_name
        if not address_full:
            address_full = address_obj.get("building_name") or address_obj.get("landmark_name") or ""
        
        # Координаты
        point = item.get("point", {})
        lon = point.get("lon", None)
        lat = point.get("lat", None)
        
        # Ссылка на 2GIS
        url_2gis = build_2gis_url(item)
        
        # Рубрики
        rubrics = item.get("rubrics", [])
        rubric_names = ", ".join([r.get("name", "") for r in rubrics])
        
        # К какому объекту относится (org) + building_name из адреса как доп. сигнал
        org = item.get("org", {}) or {}
        parent_org = org.get("name", "") or address_obj.get("building_name", "")
        
        # Расписание
        schedule_data = parse_schedule(item.get("schedule"))
        
        # Атрибуты (тарифы, инфраструктура) — реальная структура с тегами
        attr_groups = item.get("attribute_groups", [])
        tariff_data = parse_tariff(attr_groups)
        
        # Вместимость — РЕАЛЬНАЯ структура: не в самом объекте, а в links.parking[].capacity
        # (это сам объект ссылается на себя/связанные парковки через links)
        parking_spaces = None
        links = item.get("links", {}) or {}
        for p in links.get("parking", []):
            cap = p.get("capacity")
            if cap is not None:
                try:
                    parking_spaces = int(cap)
                except (ValueError, TypeError):
                    pass
                break
        # Если is_paid не удалось определить через атрибуты — берём из links.parking
        if tariff_data["is_paid"] is None:
            for p in links.get("parking", []):
                if "is_paid" in p:
                    tariff_data["is_paid"] = p["is_paid"]
                    break
        
        # Тип парковки
        parking_category = detect_parking_category(name + " " + parent_org, rubrics)
        
        # Контакты — 2GIS отдаёт это через contact_groups, если поле запрошено и есть данные
        contact_groups = item.get("contact_groups", [])
        phones = []
        websites = []
        for cg in contact_groups:
            for contact in cg.get("contacts", []):
                ctype = contact.get("type", "")
                cval = contact.get("value", "")
                if ctype == "phone":
                    phones.append(cval)
                elif ctype in ("website", "url"):
                    websites.append(cval)
        
        # Рейтинг — РЕАЛЬНАЯ структура: reviews.general_rating / general_review_count,
        # а не reviews.rating / reviews.count как предполагалось раньше
        reviews = item.get("reviews", {}) or {}
        rating = reviews.get("general_rating", None) or reviews.get("org_rating", None)
        reviews_count = reviews.get("general_review_count", None) or reviews.get("org_review_count", None)
        
        # Описание — у parking-объектов обычно нет, но проверяем на всякий случай
        description = item.get("description", "") or ""
        
        row = {
            # === ИДЕНТИФИКАЦИЯ ===
            "id_2gis": item_id,
            "name": full_name or name,
            
            # === АДРЕС ===
            "address_full": address_full,
            "street": street,
            "building": building,
            "district": district,
            
            # === КООРДИНАТЫ ===
            "longitude": lon,
            "latitude": lat,
            "coordinates": f"{lat},{lon}" if lat and lon else "",
            
            # === ССЫЛКА ===
            "url_2gis": url_2gis,
            
            # === КЛАССИФИКАЦИЯ ===
            "parking_category": parking_category,   # ТЦ / БЦ / ЖК / Городская...
            "rubrics": rubric_names,
            "parent_object": parent_org,             # к какому объекту относится
            
            # === ПЛАТНОСТЬ И ТАРИФЫ ===
            "is_paid": tariff_data["is_paid"],
            "tariff_raw": tariff_data["tariff_raw"],
            "price_hourly_tenge": tariff_data["price_hourly"],
            "price_daily_tenge": tariff_data["price_daily"],
            "price_monthly_tenge": tariff_data["price_monthly"],
            "first_hour_free": tariff_data["first_hour_free"],
            "payment_methods": tariff_data["payment_methods"],
            
            # === ПАРАМЕТРЫ ===
            "parking_spaces": parking_spaces,
            "parking_type": tariff_data["parking_type"],   # наземная/подземная/многоуровневая
            "covered": tariff_data["covered"],
            "heated": tariff_data["heated"],
            "security": tariff_data["security"],
            "video_surveillance": tariff_data["video_surveillance"],
            "disabled_spaces": tariff_data["disabled_spaces"],
            "ev_charging": tariff_data["ev_charging"],
            
            # === РАСПИСАНИЕ ===
            "hours_raw": schedule_data["hours_raw"],
            "is_24h": schedule_data["is_24h"],
            "works_on_weekends": schedule_data["works_on_weekends"],
            
            # === КОНТАКТЫ ===
            "phones": ", ".join(phones),
            "websites": ", ".join(websites),
            
            # === РЕЙТИНГ ===
            "rating": rating,
            "reviews_count": reviews_count,
            
            # === ПРОЧЕЕ ===
            "description": description,
            "parsed_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
        }
        rows.append(row)
    
    except Exception as e:
        print(f"⚠️ Ошибка обработки {item_id}: {e}")
        continue

df = pd.DataFrame(rows)
print(f"\n✅ DataFrame: {len(df)} строк, {len(df.columns)} колонок")
df.head(3)

Нормализация:   0%|          | 0/609 [00:00<?, ?it/s]

Нормализация: 100%|██████████| 609/609 [00:00<00:00, 16015.72it/s]


✅ DataFrame: 609 строк, 37 колонок


,id_2gis,name,address_full,street,building,district,longitude,latitude,coordinates,url_2gis,...,ev_charging,hours_raw,is_24h,works_on_weekends,phones,websites,rating,reviews_count,description,parsed_at
0,9429940001609611,Автостоянка,"улица Жуалы, 133",улица Жуалы,133,,76.794147,43.224093,"43.224093,76.794147",https://2gis.kz/almaty/firm/9429940001609611/7...,...,None,Пн: 24ч | Вт: 24ч | Ср: 24ч | Чт: 24ч | Пт: 24...,True,True,,,4.9,2.0,,2026-06-16 21:38
1,70000001110996268,Парковка,,,,с. Иргели,76.774963,43.231490,"43.23149,76.774963",https://2gis.kz/almaty/firm/70000001110996268/...,...,None,Пн: 24ч | Вт: 10:00–20:00 | Ср: 10:00–20:00 | ...,False,True,,,1.1,3.0,,2026-06-16 21:38
2,70000001114202109,Парковка,,,,с. Иргели,76.773588,43.231199,"43.231199,76.773588",https://2gis.kz/almaty/firm/70000001114202109/...,...,None,Пн: 24ч | Вт: 24ч | Ср: 24ч | Чт: 24ч | Пт: 24...,True,True,,,1.4,7.0,,2026-06-16 21:38


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 609 entries, 0 to 608
Data columns (total 37 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_2gis              609 non-null    object 
 1   name                 609 non-null    object 
 2   address_full         609 non-null    object 
 3   street               609 non-null    object 
 4   building             609 non-null    object 
 5   district             609 non-null    object 
 6   longitude            609 non-null    float64
 7   latitude             609 non-null    float64
 8   coordinates          609 non-null    object 
 9   url_2gis             609 non-null    object 
 10  parking_category     609 non-null    object 
 11  rubrics              609 non-null    object 
 12  parent_object        609 non-null    object 
 13  is_paid              151 non-null    object 
 14  tariff_raw           609 non-null    object 
 15  price_hourly_tenge   56 non-null     flo

In [73]:
df["has_address"] = (
    df["address_full"]
    .str.strip()
    .ne("")
)

In [74]:
# Фильтруем мусорные строки без адреса и координат

df = df.drop(
    columns=[
        "security",
        "video_surveillance",
        "disabled_spaces",
        "ev_charging"
    ],
    errors="ignore"
)

## 🧹 Очистка и обогащение данных

In [75]:
# 1. Удаляем дубли по id (на случай пересечений в сетке)
before = len(df)
df = df.drop_duplicates(subset="id_2gis").reset_index(drop=True)
print(f"Дедупликация: {before} → {len(df)} строк")

# 2. Удаляем строки без названия и без координат — явный мусор
df = df[df["name"].notna() & (df["name"] != "")]
df = df[df["longitude"].notna() & df["latitude"].notna()]
print(f"После фильтрации мусора: {len(df)} строк")

# 3. Нормализуем булевы поля → 'Да' / 'Нет' / '' для читаемости в Google Sheets
bool_columns = [
    "is_paid",
    "first_hour_free",
    "covered",
    "heated",
    "is_24h",
    "works_on_weekends"
]

for col in bool_columns:
    df[col] = (
        df[col]
        .map({True: "Да", False: "Нет"})
        .fillna("")
    )

# 4. Округляем координаты до 6 знаков
df["longitude"] = df["longitude"].round(6)
df["latitude"] = df["latitude"].round(6)

# 5. Числовые поля — заполняем пустоты
df["parking_spaces"] = df["parking_spaces"].astype("Int64")  # nullable int
df["price_hourly_tenge"] = df["price_hourly_tenge"].astype("Int64")
df["price_daily_tenge"] = df["price_daily_tenge"].astype("Int64")
df["price_monthly_tenge"] = df["price_monthly_tenge"].astype("Int64")

# 6. Сортируем по количеству заполненных полей

df["completeness"] = (
    df.replace("", pd.NA)
      .notna()
      .sum(axis=1)
)

df = (
    df.sort_values("completeness", ascending=False)
      .drop(columns=["completeness"])
      .reset_index(drop=True)
)


print(f"\n📊 Финальный датасет: {len(df)} парковок")
print("\nРаспределение по типам:")
print(df["parking_category"].value_counts())
print("\nЗаполненность платности:")
print(df["is_paid"].value_counts())

Дедупликация: 609 → 609 строк
После фильтрации мусора: 609 строк

📊 Финальный датасет: 609 парковок

Распределение по типам:
parking_category
Частная/иная    545
ТЦ/ТРЦ           46
БЦ               12
Аэропорт          5
ЖК                1
Name: count, dtype: int64

Заполненность платности:
is_paid
       458
Да      91
Нет     60
Name: count, dtype: int64


## 💾 Сохраняем CSV локально (резервная копия)

In [76]:
csv_filename = f"almaty_parkings_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8-sig")  # utf-8-sig для Excel
print(f"✅ Сохранено: {csv_filename}")

# Сохраняем сырые данные на всякий случай
with open("raw_2gis_data.json", "w", encoding="utf-8") as f:
    json.dump(list(all_raw.values()), f, ensure_ascii=False, indent=2)
print("✅ Сырые данные: raw_2gis_data.json")

✅ Сохранено: almaty_parkings_20260616_2143.csv
✅ Сырые данные: raw_2gis_data.json


## 📊 Выгрузка в Google Sheets

Используем `gspread` + Service Account. Документация: [gspread.readthedocs.io](https://docs.gspread.org/)

In [77]:
import gspread
from google.oauth2.service_account import Credentials
import numpy as np

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

try:
    creds = Credentials.from_service_account_file(GOOGLE_CREDENTIALS_FILE, scopes=SCOPES)
    gc = gspread.authorize(creds)
    print("✅ Google авторизация успешна")
except FileNotFoundError:
    print(f"❌ Файл {GOOGLE_CREDENTIALS_FILE} не найден. Смотри инструкцию в начале ноутбука.")
    raise
except Exception as e:
    print(f"❌ Ошибка авторизации: {e}")
    raise

✅ Google авторизация успешна


In [78]:
def prepare_for_sheets(df: pd.DataFrame) -> list[list]:
    """Конвертируем DataFrame в список списков для gspread.
    Google Sheets не принимает NaN, pd.NA, numpy типы."""
    
    # Заголовки на русском для читаемости таблицы
    headers_ru = {
        "id_2gis": "ID в 2GIS",
        "name": "Название",
        "address_full": "Адрес полный",
        "street": "Улица",
        "building": "Дом",
        "district": "Район",
        "longitude": "Долгота",
        "latitude": "Широта",
        "coordinates": "Координаты (lat,lon)",
        "url_2gis": "Ссылка на 2GIS",
        "parking_category": "Тип (ТЦ/БЦ/Городская...)",
        "rubrics": "Рубрики 2GIS",
        "parent_object": "Относится к объекту",
        "is_paid": "Платная?",
        "tariff_raw": "Тариф (сырой)",
        "price_hourly_tenge": "Цена за час (₸)",
        "price_daily_tenge": "Цена за сутки (₸)",
        "price_monthly_tenge": "Абонемент/мес (₸)",
        "first_hour_free": "Первый час бесплатно?",
        "payment_methods": "Способы оплаты",
        "parking_spaces": "Кол-во мест",
        "parking_type": "Тип (наземная/подземная)",
        "covered": "Крытая?",
        "heated": "Тёплая?",
        "security": "Охраняемая?",
        "video_surveillance": "Видеонаблюдение?",
        "disabled_spaces": "Места для инвалидов?",
        "ev_charging": "Зарядка для ЭМ?",
        "hours_raw": "Часы работы",
        "is_24h": "Круглосуточно?",
        "works_on_weekends": "Работает в выходные?",
        "phones": "Телефоны",
        "websites": "Сайты",
        "rating": "Рейтинг 2GIS",
        "reviews_count": "Кол-во отзывов",
        "description": "Описание",
        "parsed_at": "Дата парсинга",
    }
    
    header_row = [headers_ru.get(col, col) for col in df.columns]
    
    def clean_val(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return ""
        if isinstance(v, (np.integer, np.int64)):
            return int(v)
        if isinstance(v, (np.floating, np.float64)):
            return float(v)
        if pd.isna(v) if not isinstance(v, str) else False:
            return ""
        return str(v) if not isinstance(v, (int, float, bool)) else v
    
    data_rows = []
    for _, row in df.iterrows():
        data_rows.append([clean_val(v) for v in row])
    
    return [header_row] + data_rows


print("✅ Функция подготовки данных для Sheets готова")

✅ Функция подготовки данных для Sheets готова


In [79]:
# Открываем или создаём таблицу
try:
    spreadsheet = gc.open(GOOGLE_SHEET_NAME)
    print(f"📄 Открыта существующая таблица: {GOOGLE_SHEET_NAME}")
except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(GOOGLE_SHEET_NAME)
    print(f"📄 Создана новая таблица: {GOOGLE_SHEET_NAME}")
    # Делаем таблицу доступной по ссылке (для ревьюеров)
    spreadsheet.share(None, perm_type="anyone", role="reader")
    print("🔓 Доступ 'по ссылке — просмотр' выдан")

# Работаем с первым листом
worksheet = spreadsheet.sheet1
worksheet.clear()

# Подготавливаем данные
sheet_data = prepare_for_sheets(df)
print(f"📊 Загружаем {len(sheet_data)-1} строк × {len(sheet_data[0])} колонок...")

# Загружаем батчами по 1000 строк (лимит gspread)
BATCH_SIZE = 1000
worksheet.update(sheet_data[:BATCH_SIZE+1])  # заголовок + первый батч

for i in range(BATCH_SIZE+1, len(sheet_data), BATCH_SIZE):
    batch = sheet_data[i:i+BATCH_SIZE]
    start_row = i + 1  # 1-indexed
    worksheet.append_rows(batch, value_input_option="USER_ENTERED")
    print(f"  Загружен батч {i}–{i+len(batch)-1}")
    time.sleep(1)

print(f"\n✅ Данные выгружены!")
print(f"🔗 Ссылка: {spreadsheet.url}")

📄 Открыта существующая таблица: 2GIS Parking Almaty


C:\Users\User\AppData\Local\Temp\ipykernel_9192\2910064441.py:55: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(v) if not isinstance(v, str) else False:


📊 Загружаем 609 строк × 34 колонок...

✅ Данные выгружены!
🔗 Ссылка: https://docs.google.com/spreadsheets/d/1dta21MhyH-1TToDa5JiEk_AeoIePwMcUphJJqpCNtWc


## 📈 Статистика по датасету

In [80]:
print("=" * 50)
print("📊 ИТОГОВАЯ СТАТИСТИКА")
print("=" * 50)
print(f"Всего парковок: {len(df)}")
print(f"Уникальных ID: {df['id_2gis'].nunique()}")
print()
print("По типу:")
print(df["parking_category"].value_counts().to_string())
print()
print("Платность:")
print(df["is_paid"].value_counts().to_string())
print()
print("Круглосуточных:", (df["is_24h"] == "Да").sum())
print("С рейтингом:", df["rating"].notna().sum())
print("Средний рейтинг:", df["rating"].mean().round(2) if df["rating"].notna().any() else "нет данных")
print()
print("Заполненность ключевых полей:")
key_fields = ["address_full", "coordinates", "url_2gis", "parking_category",
              "is_paid", "hours_raw", "parking_spaces"]
for f in key_fields:
    filled = (df[f].notna() & (df[f] != "")).sum()
    pct = filled / len(df) * 100
    print(f"  {f}: {filled}/{len(df)} ({pct:.0f}%)")

📊 ИТОГОВАЯ СТАТИСТИКА
Всего парковок: 609
Уникальных ID: 609

По типу:
parking_category
Частная/иная    545
ТЦ/ТРЦ           46
БЦ               12
Аэропорт          5
ЖК                1

Платность:
is_paid
       458
Да      91
Нет     60

Круглосуточных: 189
С рейтингом: 400
Средний рейтинг: 2.56

Заполненность ключевых полей:
  address_full: 95/609 (16%)
  coordinates: 609/609 (100%)
  url_2gis: 609/609 (100%)
  parking_category: 609/609 (100%)
  is_paid: 151/609 (25%)
  hours_raw: 608/609 (100%)
  parking_spaces: 17/609 (3%)


In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 609 entries, 0 to 608
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id_2gis              609 non-null    object 
 1   name                 609 non-null    object 
 2   address_full         609 non-null    object 
 3   street               609 non-null    object 
 4   building             609 non-null    object 
 5   district             609 non-null    object 
 6   longitude            609 non-null    float64
 7   latitude             609 non-null    float64
 8   coordinates          609 non-null    object 
 9   url_2gis             609 non-null    object 
 10  parking_category     609 non-null    object 
 11  rubrics              609 non-null    object 
 12  parent_object        609 non-null    object 
 13  is_paid              609 non-null    object 
 14  tariff_raw           609 non-null    object 
 15  price_hourly_tenge   56 non-null     Int